In [4]:
# Import necessary libraries and functions
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests
from sklearn.decomposition import PCA
from scipy.stats import entropy

In [3]:
# Dataset creation for post-hoc comparisons 

# Load the dataset
file_path = '/home/cerna3/neuroconn/data analyses/final_combined_data_compressed.xlsx'
data = pd.read_excel(file_path)

# Function to parse connectivity values, handling within/between and EC/EO
def parse_connectivity(data, key, suffix, is_between=False):
    records = []
    for idx, row in data.iterrows():
        group = row['Group']
        mode = row['Mode']
        conn_dict = eval(row[key])

        for networks, states in conn_dict.items():
            if is_between:
                networks = ','.join(networks)  # Combine network pair into single string
            for state, value in states.items():
                records.append({
                    'ID': row['ID'],
                    'Group': group,
                    'Mode': mode,
                    'Network' if not is_between else 'Network_Pair': networks,
                    'State': state,
                    'Value': value
                })
    return pd.DataFrame(records)

# Function to parse transition magnitudes
def parse_transition_magnitudes(data, key, suffix, is_between=False):
    records = []
    for idx, row in data.iterrows():
        group = row['Group']
        mode = row['Mode']
        tran_mag_dict = eval(row[key])
        for networks, states in tran_mag_dict.items():
            if is_between:
                networks = ','.join(networks) 
            for state, value in states.items():
                if isinstance(value, list):
                    value = value[0] if value else None
                records.append({
                    'ID': row['ID'],
                    'Group': group,
                    'Mode': mode,
                    'Network' if not is_between else 'Network_Pair': networks,
                    'State': state,
                    'Value': value
                })
    
    return pd.DataFrame(records)

def add_temporal_variables(data, temporal_vars, var_name):
    records = []
    for idx, row in data.iterrows():
        mode = row['Mode']
        var_values = temporal_vars[idx]
        
        if mode == 'EC' and len(var_values) >= 7:
            for state, value in enumerate(var_values[:7]):
                records.append({
                    'ID': row['ID'],
                    'Group': row['Group'],
                    'Mode': mode,
                    'State': state,
                    f'{var_name}': value
                })
        elif mode == 'EO' and len(var_values) >= 5:
            for state, value in enumerate(var_values[:5]):
                records.append({
                    'ID': row['ID'],
                    'Group': row['Group'],
                    'Mode': mode,
                    'State': state,
                    f'{var_name}': value
                })
    
    return pd.DataFrame(records)

def add_transition_probabilities(data, transition_probs):
    records = []
    
    for idx, row in data.iterrows():
        mode = row['Mode']
        id = row['ID']
        group = row['Group']
        trans_probs = transition_probs[idx]
        
        if not isinstance(trans_probs, list) or not trans_probs:
            print(f"Warning: Unexpected format for transition probabilities for ID {id}, Mode {mode}")
            continue
        
        # Extract the inner list
        if len(trans_probs) == 1 and isinstance(trans_probs[0], list):
            trans_probs = trans_probs[0]
        
        matrix_size = 7 if mode == 'EC' else 5
        if len(trans_probs) != matrix_size * matrix_size:
            print(f"Warning: Insufficient data for ID {id}, Mode {mode}. Length: {len(trans_probs)}")
            continue
        
        # Reshape into matrix
        trans_probs_matrix = np.array(trans_probs).reshape(matrix_size, matrix_size)
        
        # Perform PCA
        pca = PCA(n_components=1)
        pca_result = pca.fit_transform(trans_probs_matrix)
        
        # Calculate entropy for each row (state)
        entropies = [entropy(row) for row in trans_probs_matrix]
        
        for state in range(matrix_size):
            records.append({
                'ID': id,
                'Group': group,
                'Mode': mode,
                'State': state,
                'Transition_Probabilities_PCA': pca_result[state][0],
                'Transition_Probabilities_Entropy': entropies[state]
            })
    
    result_df = pd.DataFrame(records)
    if result_df.empty:
        print("Warning: No transition probabilities were processed.")
    else:
        print(f"Processed {len(result_df)} transition probability records.")
    return result_df

# --- Data extraction and parsing ---
within_conn_mean_df = parse_connectivity(data, 'Within_Network_Conn_Mean', '_w_mean')
within_tran_mag_df = parse_transition_magnitudes(data, 'Within_Network_Transition_Magnitudes', '_w_tran_mag')
between_conn_mean_df = parse_connectivity(data, 'Between_Network_Conn_Mean', '_b_mean', is_between=True)
between_tran_mag_df = parse_transition_magnitudes(data, 'Between_Network_Transition_Magnitudes', '_b_tran_mag', is_between=True)

# --- Temporal variable extraction and processing ---
def format_temporal_var(column, is_transition_prob=False):
    def format_single_value(x):
        if is_transition_prob:
            # Remove all brackets and split by spaces
            values = x.replace('[', '').replace(']', '').split()
            return [float(val) for val in values if val.replace('.', '').isdigit()]
        else:
            # For other temporal variables, keep the existing logic
            values = x.strip('[]').split()
            return [float(val) for val in values if val.replace('.', '').isdigit()]
    
    return column.apply(format_single_value)

# Modify the temporal_vars extraction:
temporal_vars = {
    'Transition_Probabilities': format_temporal_var(data['Transition_Probabilities'], is_transition_prob=True).tolist(),
    'Mean_Lifetime': format_temporal_var(data['Mean_Lifetime']).tolist(),
    'Fractional_Occupancy': format_temporal_var(data['Fractional_Occupancy']).tolist(),
    'Mean_Interval_Length': format_temporal_var(data['Mean_Interval_Length']).tolist(),
}

# Create temporal dataframes
mean_lifetime_df = add_temporal_variables(data, temporal_vars['Mean_Lifetime'], 'Mean_Lifetime')
fractional_occupancy_df = add_temporal_variables(data, temporal_vars['Fractional_Occupancy'], 'Fractional_Occupancy')
mean_interval_length_df = add_temporal_variables(data, temporal_vars['Mean_Interval_Length'], 'Mean_Interval_Length')
transition_probabilities_df = add_transition_probabilities(data, temporal_vars['Transition_Probabilities'])

# Combine all temporal dataframes
temporal_df = mean_lifetime_df.merge(fractional_occupancy_df, on=['ID', 'Group', 'Mode', 'State'], how='outer')
temporal_df = temporal_df.merge(mean_interval_length_df, on=['ID', 'Group', 'Mode', 'State'], how='outer')
temporal_df = temporal_df.merge(transition_probabilities_df, on=['ID', 'Group', 'Mode', 'State'], how='outer')

def remove_temporal_duplicates(df):
    temporal_vars = ['Mean_Lifetime', 'Fractional_Occupancy', 'Mean_Interval_Length', 
                     'Transition_Probabilities_PCA', 'Transition_Probabilities_Entropy']
    
    for var in temporal_vars:
        if var not in df.columns:
            print(f"Column {var} not found in DataFrame. Skipping...")
            continue
        
        # Sort the dataframe by ID, Group, Mode, State, and the temporal variable
        df = df.sort_values(['ID', 'Group', 'Mode', 'State', var])
        
        # Remove duplicates, keeping the first occurrence
        df = df.drop_duplicates(subset=['ID', 'Group', 'Mode', 'State', var], keep='first')
    
    return df

# --- Pivot spatial data to wide format for easier analysis ---
def pivot_data(df, suffix):
    if suffix in ['_w_mean', '_w_tran_mag']:
        # For within-network data
        pivoted = df.pivot_table(
            index=['ID', 'Group', 'Mode', 'State'],
            columns='Network',
            values='Value'
        ).reset_index()
        
        # Rename columns
        pivoted.columns.name = None
        pivoted.rename(columns={col: f"{col}{suffix}" for col in pivoted.columns if col not in ['ID', 'Group', 'Mode', 'State']}, inplace=True)
    else:
        # For between-network data
        pivoted = df.pivot_table(
            index=['ID', 'Group', 'Mode', 'State'],
            columns='Network_Pair',
            values='Value'
        ).reset_index()
        
        # Rename columns, replacing comma with underscore
        pivoted.columns.name = None
        pivoted.rename(columns={col: f"{col.replace(', ', '_')}{suffix}" for col in pivoted.columns if col not in ['ID', 'Group', 'Mode', 'State']}, inplace=True)
    
    return pivoted

# --- Pivot spatial variables ---
within_conn_mean_pivot = pivot_data(within_conn_mean_df, '_w_mean')
within_tran_mag_pivot = pivot_data(within_tran_mag_df, '_w_tran_mag')
between_conn_mean_pivot = pivot_data(between_conn_mean_df, '_b_mean')
between_tran_mag_pivot = pivot_data(between_tran_mag_df, '_b_tran_mag')

# Merge spatial dataframes
spatial_df = (
    within_conn_mean_pivot
    .merge(within_tran_mag_pivot, on=['ID', 'Group', 'Mode', 'State'])
    .merge(between_conn_mean_pivot, on=['ID', 'Group', 'Mode', 'State'])
    .merge(between_tran_mag_pivot, on=['ID', 'Group', 'Mode', 'State'])
)

# Load the original dataset again (if needed)
original_data_path = '/home/cerna3/neuroconn/data analyses/final_combined_data_compressed.xlsx'
original_data = pd.read_excel(original_data_path)

# Extract final_score_miniBEST, Practice, and Age from the original dataset
original_variables = original_data[['ID', 'final_score_miniBEST', 'Practice', 'Age']]

# Merge spatial and temporal dataframes
combined_df = spatial_df.merge(temporal_df, on=['ID', 'Group', 'Mode', 'State'], how='outer')

# Merge final_score_miniBEST, Practice, and Age into the new dataset
combined_df_with_additional_info = pd.merge(combined_df, original_variables, on='ID', how='left')

# Remove duplicates from temporal variables
combined_df_with_additional_info = remove_temporal_duplicates(combined_df_with_additional_info)

# --- Save and display ---
output_file_path = '/home/cerna3/neuroconn/data analyses/combined_network_data.xlsx'
combined_df_with_additional_info.to_excel(output_file_path, index=False)

print(combined_df_with_additional_info.info())

Processed 540 transition probability records.
<class 'pandas.core.frame.DataFrame'>
Index: 540 entries, 0 to 1078
Data columns (total 68 columns):
 #   Column                                                                       Non-Null Count  Dtype  
---  ------                                                                       --------------  -----  
 0   ID                                                                           540 non-null    int64  
 1   Group                                                                        540 non-null    object 
 2   Mode                                                                         540 non-null    object 
 3   State                                                                        540 non-null    int64  
 4   Default_w_mean                                                               540 non-null    float64
 5   DorsalAttention_w_mean                                                       540 non-null    float64
 6   

In [4]:
# Mann-Whitney U with Winsorization

import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests

def winsorize_series_to_iqr_bounds(data_series):
    """
    Winsorize a series by clipping values to the 1.5 * IQR bounds.
    
    Parameters:
    data_series: pandas Series of values
    
    Returns:
    winsorized_series: pandas Series with outliers clipped to IQR bounds
    """
    if not isinstance(data_series, pd.Series):
        # Ensure it's a Series, especially if it's a slice that might be a DataFrame
        if isinstance(data_series, pd.DataFrame) and data_series.shape[1] == 1:
            data_series = data_series.iloc[:, 0]
        else:
            raise ValueError("Input must be a pandas Series or a single-column DataFrame.")

    series_copy = data_series.copy()
    
    if len(series_copy.dropna()) < 3: # Need at least 3 non-NA values to compute IQR robustly
        return series_copy
        
    Q1 = series_copy.quantile(0.25)
    Q3 = series_copy.quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Clip values to the bounds
    # Only apply if IQR is not zero (to avoid issues with constant data)
    if IQR > 0:
        series_copy = series_copy.clip(lower=lower_bound, upper=upper_bound)
    
    return series_copy

def apply_general_iqr_winsorization(data, group_column, value_columns):
    """
    Apply IQR-based winsorization to specified value columns, grouped by a group_column.
    
    Parameters:
    data: DataFrame containing the data.
    group_column: Name of the column containing group labels.
    value_columns: List of column names to winsorize.
    
    Returns:
    data_winsorized: DataFrame with specified columns winsorized within each group.
    """
    data_winsorized = data.copy()
    
    print(f"\nApplying general IQR winsorization across groups for {len(value_columns)} variables...")
    
    for col_to_winsorize in value_columns:
        if col_to_winsorize in data_winsorized.columns:
            # Use transform to apply winsorization within each group and align results
            # Ensure that NaN values are handled correctly by winsorize_series_to_iqr_bounds
            # or by handling them before/after grouping if necessary.
            # The current winsorize_series_to_iqr_bounds should handle NaNs within the Series.
            data_winsorized[col_to_winsorize] = data_winsorized.groupby(group_column)[col_to_winsorize]\
                                                               .transform(winsorize_series_to_iqr_bounds)
            print(f"  Winsorized: {col_to_winsorize}")
        else:
            print(f"  Warning: Column {col_to_winsorize} not found for winsorization. Skipping.")
            
    print("General IQR winsorization completed.")
    return data_winsorized

def average_across_modes_states(data, connectivity_columns):
    """
    Average connectivity values across modes and states for each participant.
    
    Parameters:
    data: DataFrame containing the data with multiple rows per participant
    connectivity_columns: List of column names to average
    
    Returns:
    averaged_data: DataFrame with one row per participant containing averaged values
    """
    print("Averaging data across modes and states for each participant...")
    
    # Ensure ID column is string type for consistent comparison
    data['ID'] = data['ID'].astype(str)
    
    # Group by ID and Group, then calculate the mean for each connectivity variable
    grouping_columns = ['ID', 'Group']
    
    # Check if there are additional grouping variables we should preserve
    non_connectivity_cols = [col for col in data.columns if col not in connectivity_columns]
    
    print(f"Grouping by: {grouping_columns}")
    print(f"Averaging {len(connectivity_columns)} connectivity variables")
    
    # Create aggregation dictionary
    agg_dict = {}
    
    # For connectivity columns, calculate mean
    for col in connectivity_columns:
        if col in data.columns:
            agg_dict[col] = 'mean'
    
    # For non-connectivity columns that are not part of the primary grouping, take first value
    # (assuming they're the same within each participant after initial ID/Group grouping)
    for col in non_connectivity_cols:
        if col not in grouping_columns and col in data.columns:
            # Check if the column is constant within each ID-Group pair before deciding 'first'
            # For simplicity here, we'll stick to 'first' as per original script logic for non-numeric/meta_cols
            agg_dict[col] = 'first' 
            
    # Perform the aggregation
    averaged_data = data.groupby(grouping_columns, as_index=False).agg(agg_dict)
    
    # Report the averaging results
    original_rows = len(data)
    averaged_rows = len(averaged_data)
    
    print(f"Original data: {original_rows} rows")
    print(f"Averaged data: {averaged_rows} rows")
    if averaged_rows > 0:
        print(f"Average rows per participant (before averaging): {original_rows/averaged_rows:.2f}")
    else:
        print("No data after averaging.")
        
    # Show participant counts by group
    participant_counts = averaged_data['Group'].value_counts()
    print(f"Participant counts by group after averaging:")
    for group, count in participant_counts.items():
        print(f"  {group}: {count} participants")
        
    return averaged_data

# Load the data from the provided Excel file
# You'll need to replace this with the correct path if you run this locally
# For the AI environment, it assumes the file is in the root.
try:
    file_path = 'combined_network_data.xlsx' # User needs to upload this file
    data_raw = pd.read_excel(file_path)
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please ensure it is uploaded and accessible.")
    # Create a dummy DataFrame to allow the script to run without crashing for demonstration
    data_raw = pd.DataFrame({
        'ID': [f'P{i//3}{g}' for i in range(30) for g in ['YACs', 'OACs', 'TCOAs'][:1+(i%3)]],
        'Group': [g for i in range(30) for g in ['YACs', 'OACs', 'TCOAs'][:1+(i%3)]],
        'Default_w_mean': np.random.rand(90) * 0.1,
        'Visual_w_mean': np.random.rand(90) * 0.1,
        'Default_Visual_b_mean': np.random.rand(90) * 0.05,
        'Mean_Lifetime': np.random.rand(90) + 1,
        'Fractional_Occupancy': np.random.rand(90),
        'Mean_Interval_Length': np.random.rand(90) * 0.5,
        'Transition_Probabilities_PCA': np.random.rand(90),
        'Transition_Probabilities_Entropy': np.random.rand(90),
        'Some_Other_Column': ['A'] * 90 # Example of a non-connectivity column
    })
    # Add some extreme values to test winsorization
    if not data_raw.empty:
        data_raw.loc[0, 'Default_w_mean'] = 1.0 # Extreme value for YACs
        data_raw.loc[30, 'Default_w_mean'] = 1.2 # Extreme value for OACs
        data_raw.loc[60, 'Default_w_mean'] = -0.5 # Extreme value for TCOAs
    print("Using dummy data as 'combined_network_data.xlsx' was not found.")


data = data_raw.copy() # Work with a copy

# Ensure ID column is string type for consistent comparison
data['ID'] = data['ID'].astype(str)

print(f"Original dataset: {len(data)} rows, {len(data['ID'].unique())} unique participants")

# Aggregate connectivity variables from the original dataset
within_network_columns = [col for col in data.columns if '_w_mean' in col]
within_network_tran_mag_columns = [col for col in data.columns if '_w_tran_mag' in col]
between_network_columns = [col for col in data.columns if '_b_mean' in col]
between_network_tran_mag_columns = [col for col in data.columns if '_b_tran_mag' in col]

# Include temporal variables
temporal_columns = ['Mean_Lifetime', 'Fractional_Occupancy', 'Mean_Interval_Length', 
                    'Transition_Probabilities_PCA', 'Transition_Probabilities_Entropy']

# Ensure only existing columns are included
all_potential_columns = (within_network_columns + within_network_tran_mag_columns + 
                         between_network_columns + between_network_tran_mag_columns + temporal_columns)
connectivity_columns = [col for col in all_potential_columns if col in data.columns]


print(f"Total connectivity variables identified: {len(connectivity_columns)}")
if not connectivity_columns:
    print("Warning: No connectivity columns found based on naming patterns. Check column names and patterns.")

# Average across modes and states instead of dropping duplicates
data_averaged = average_across_modes_states(data, connectivity_columns)

def perform_mann_whitney_comparisons(data_input, group_column, conn_cols, analysis_name=""):
    """
    Perform Mann-Whitney U tests between groups for all connectivity variables.
    """
    results = []
    df_to_test = data_input.copy() # Work on a copy
    
    print(f"\nPerforming Mann-Whitney U comparisons for {analysis_name}...")
    if df_to_test.empty or group_column not in df_to_test.columns:
        print(f"Warning: Data for {analysis_name} is empty or group column missing. Skipping.")
        return pd.DataFrame()
        
    print(f"Groups in data: {df_to_test[group_column].unique()}")
    print(f"Sample sizes: {df_to_test[group_column].value_counts().to_dict()}")
    
    for column in conn_cols:
        if column not in df_to_test.columns:
            print(f"Warning: Column {column} not found in data for {analysis_name}. Skipping.")
            continue
            
        levels = sorted(df_to_test[group_column].unique()) # Sort for consistent comparison order
        # Ensure there are at least two groups to compare
        if len(levels) < 2:
            print(f"Warning: Fewer than two groups found for column {column} in {analysis_name}. Skipping.")
            continue

        # Define pairs for comparison (e.g., YACs vs OACs, OACs vs TCOAs)
        # This part might need adjustment based on desired specific comparisons
        comparisons = []
        if 'YACs' in levels and 'OACs' in levels:
            comparisons.append(('YACs', 'OACs'))
        if 'OACs' in levels and 'TCOAs' in levels:
            comparisons.append(('OACs', 'TCOAs'))
        # Add more specific pairs if needed, e.g. YACs vs TCOAs
        if 'YACs' in levels and 'TCOAs' in levels and ('YACs', 'TCOAs') not in comparisons:
             if analysis_name.startswith("Practice Comparison"): # Typically OACs vs TCOAs
                 pass # Don't add YACs vs TCOAs for practice comparison
             elif analysis_name.startswith("Age Comparison"): # Typically YACs vs OACs
                 pass # Don't add YACs vs TCOAs for age comparison unless specified
             else: # For a general case, might include it.
                 comparisons.append(('YACs', 'TCOAs'))


        for i, j in comparisons:
            # Check if this comparison is relevant for the current analysis_name
            # (e.g., age_comparison_data only has YACs, OACs)
            if not (data_input[group_column].isin([i,j]).all()):
                 # This check might be too restrictive if data_input is pre-filtered
                 # The filtering before calling this function should handle group relevance.
                 pass


            group_i_data = df_to_test[df_to_test[group_column] == i][column].dropna()
            group_j_data = df_to_test[df_to_test[group_column] == j][column].dropna()
            
            if len(group_i_data) == 0 or len(group_j_data) == 0:
                print(f"Warning: Empty group for {column} comparison {i} vs {j} in {analysis_name}. Skipping.")
                continue
            
            try:
                statistic, p_value = stats.mannwhitneyu(group_i_data, group_j_data, alternative='two-sided')
            except ValueError as e:
                print(f"Error in Mann-Whitney U for {column} ({i} vs {j}): {e}. Skipping.")
                continue

            mean_i, mean_j = group_i_data.mean(), group_j_data.mean()
            median_i, median_j = group_i_data.median(), group_j_data.median()
            
            mean_diff = mean_i - mean_j
            median_diff = median_i - median_j
            
            n1, n2 = len(group_i_data), len(group_j_data)
            if n1 * n2 == 0: # Avoid division by zero if a group somehow became empty post-dropna
                rank_biserial_correlation = np.nan
            else:
                rank_biserial_correlation = (statistic / (n1 * n2)) - ( (n1+n2+1 - (statistic / (n1*n2)) ) / (n1*n2) ) # Using U / (n1*n2) as common effect size
                # Or simpler: rank_biserial_correlation = 1 - (2 * statistic) / (n1 * n2) if U is for group1 < group2
                # A common formula for r from U is r = 1 - (2U)/(n1n2) if U is the smaller U.
                # Or using the U from stats.mannwhitneyu (which can be U1 or U2):
                # r = (U - (n1*n2/2)) / sqrt((n1*n2*(n1+n2+1))/12) -- this is Z/sqrt(N)
                # Rank Biserial r = (2 * U_statistic) / (n1 * n2) - 1  (if U is defined as number of times x_i > y_j)
                # scipy.stats.mannwhitneyu returns U for the first group.
                # So, if group_i is group1, then U is for group_i.
                # r = U / (n1*n2) - U_expected / (n1*n2) where U_expected for null is n1n2/2
                # r = (statistic - n1*n2/2) / (n1*n2/2) = 2*statistic/(n1*n2) - 1  <-- This one is commonly cited
                rank_biserial_correlation = (2 * statistic / (n1 * n2)) - 1


            results.append([i, j, column, mean_i, mean_j, mean_diff, median_i, median_j, 
                            median_diff, statistic, p_value, rank_biserial_correlation, n1, n2])
    
    results_df = pd.DataFrame(results, columns=['group1', 'group2', 'variable', 'mean1', 'mean2', 'mean_diff',
                                                'median1', 'median2', 'median_diff', 'U-statistic', 'p-value',
                                                'Rank Biserial Correlation (r)', 'n1', 'n2'])
    
    if not results_df.empty:
        reject, pvals_fdr, _, _ = multipletests(results_df['p-value'].fillna(1.0), method='fdr_bh') # fillna for safety
        results_df.insert(results_df.columns.get_loc('p-value') + 1, 'FDR-adjusted p-value', pvals_fdr)
    
    print(f"Completed {len(results_df)} comparisons for {analysis_name}")
    return results_df

# Ensure data_averaged is not empty before proceeding
if data_averaged.empty:
    print("Error: data_averaged is empty. Cannot proceed with analyses.")
    # Depending on execution context, may want to exit or raise error
else:
    print("="*80)
    print("RUNNING MANN-WHITNEY U TESTS WITH AVERAGED DATA - THREE APPROACHES")
    print("="*80)

    # 1. Original analysis (with averaging, no other modifications)
    print("\n1. ORIGINAL ANALYSIS (With averaging across modes/states)")
    age_comparison_data_original = data_averaged[data_averaged['Group'].isin(['YACs', 'OACs'])].copy()
    practice_comparison_data_original = data_averaged[data_averaged['Group'].isin(['OACs', 'TCOAs'])].copy()

    age_results_original = perform_mann_whitney_comparisons(
        age_comparison_data_original, 'Group', connectivity_columns, "Age Comparison - Averaged Data"
    )
    practice_results_original = perform_mann_whitney_comparisons(
        practice_comparison_data_original, 'Group', connectivity_columns, "Practice Comparison - Averaged Data"
    )

    # 2. Analysis without subject 401 (with averaging)
    print("\n2. ANALYSIS WITHOUT SUBJECT 401 (With averaging)")
    data_averaged_without_401 = data_averaged[data_averaged['ID'] != '401'].copy()
    
    age_comparison_data_without_401 = data_averaged_without_401[data_averaged_without_401['Group'].isin(['YACs', 'OACs'])].copy()
    practice_comparison_data_without_401 = data_averaged_without_401[data_averaged_without_401['Group'].isin(['OACs', 'TCOAs'])].copy()

    age_results_without_401 = perform_mann_whitney_comparisons(
        age_comparison_data_without_401, 'Group', connectivity_columns, "Age Comparison - Averaged, Without 401"
    )
    practice_results_without_401 = perform_mann_whitney_comparisons(
        practice_comparison_data_without_401, 'Group', connectivity_columns, "Practice Comparison - Averaged, Without 401"
    )

    # 3. Analysis with general IQR winsorization (with averaging)
    print("\n3. ANALYSIS WITH GENERAL IQR WINSORIZATION (With averaging)")
    # Apply general IQR winsorization to the averaged data
    data_averaged_iqr_winsorized = apply_general_iqr_winsorization(data_averaged, 'Group', connectivity_columns)
    
    age_comparison_data_iqr_winsorized = data_averaged_iqr_winsorized[data_averaged_iqr_winsorized['Group'].isin(['YACs', 'OACs'])].copy()
    practice_comparison_data_iqr_winsorized = data_averaged_iqr_winsorized[data_averaged_iqr_winsorized['Group'].isin(['OACs', 'TCOAs'])].copy()

    age_results_iqr_winsorized = perform_mann_whitney_comparisons(
        age_comparison_data_iqr_winsorized, 'Group', connectivity_columns, "Age Comparison - Averaged, IQR Winsorized"
    )
    practice_results_iqr_winsorized = perform_mann_whitney_comparisons(
        practice_comparison_data_iqr_winsorized, 'Group', connectivity_columns, "Practice Comparison - Averaged, IQR Winsorized"
    )

    # Save all results to Excel file with different tabs
    output_excel_file = 'mann_whitney_results_averaged_comprehensive_iqr_winsor.xlsx'
    print(f"\nSaving results to Excel file: {output_excel_file}...")
    with pd.ExcelWriter(output_excel_file) as writer:
        # Original results (with averaging)
        if not age_results_original.empty: age_results_original.to_excel(writer, sheet_name='age_avg', index=False)
        if not practice_results_original.empty: practice_results_original.to_excel(writer, sheet_name='practice_avg', index=False)
        
        # Results without 401 (with averaging)
        if not age_results_without_401.empty: age_results_without_401.to_excel(writer, sheet_name='age_avg_no401', index=False)
        if not practice_results_without_401.empty: practice_results_without_401.to_excel(writer, sheet_name='practice_avg_no401', index=False)
        
        # Results with general IQR winsorization (with averaging)
        if not age_results_iqr_winsorized.empty: age_results_iqr_winsorized.to_excel(writer, sheet_name='age_avg_iqr_winsor', index=False)
        if not practice_results_iqr_winsorized.empty: practice_results_iqr_winsorized.to_excel(writer, sheet_name='practice_avg_iqr_winsor', index=False)
        
        # Save the variously processed averaged datasets for reference
        if not data_averaged.empty: data_averaged.to_excel(writer, sheet_name='averaged_dataset_original', index=False)
        if not data_averaged_without_401.empty: data_averaged_without_401.to_excel(writer, sheet_name='averaged_dataset_no401', index=False)
        if not data_averaged_iqr_winsorized.empty: data_averaged_iqr_winsorized.to_excel(writer, sheet_name='averaged_dataset_iqr_winsor', index=False)

    print(f"Mann-Whitney U test results have been saved to '{output_excel_file}'")

    # Create summary comparison
    print("\n" + "="*80)
    print("SUMMARY COMPARISON OF SIGNIFICANT RESULTS (AVERAGED DATA)")
    print("="*80)

    def summarize_significant_results(results_df, analysis_desc, alpha=0.05):
        """Summarize significant results from Mann-Whitney U tests."""
        if results_df.empty:
            print(f"\n{analysis_desc}: No results to summarize (DataFrame is empty).")
            return {'total': 0, 'sig_uncorrected': 0, 'sig_fdr': 0}
        
        total_comparisons = len(results_df)
        sig_uncorrected = len(results_df[results_df['p-value'] < alpha])
        sig_fdr = 0
        if 'FDR-adjusted p-value' in results_df.columns:
            sig_fdr = len(results_df[results_df['FDR-adjusted p-value'] < alpha])
        
        print(f"\n{analysis_desc}:")
        print(f"  Total comparisons: {total_comparisons}")
        if total_comparisons > 0:
            print(f"  Significant (p < {alpha}): {sig_uncorrected} ({sig_uncorrected/total_comparisons*100:.1f}%)")
            print(f"  Significant (FDR < {alpha}): {sig_fdr} ({sig_fdr/total_comparisons*100:.1f}%)")
        else:
            print(f"  Significant (p < {alpha}): 0 (N/A%)")
            print(f"  Significant (FDR < {alpha}): 0 (N/A%)")
            
        return {'total': total_comparisons, 'sig_uncorrected': sig_uncorrected, 'sig_fdr': sig_fdr}

    # Summarize all analyses
    age_summary_original = summarize_significant_results(age_results_original, "Age Comparison - Averaged Data")
    practice_summary_original = summarize_significant_results(practice_results_original, "Practice Comparison - Averaged Data")

    age_summary_without_401 = summarize_significant_results(age_results_without_401, "Age Comparison - Averaged, Without 401")
    practice_summary_without_401 = summarize_significant_results(practice_results_without_401, "Practice Comparison - Averaged, Without 401")

    age_summary_iqr_winsorized = summarize_significant_results(age_results_iqr_winsorized, "Age Comparison - Averaged, IQR Winsorized")
    practice_summary_iqr_winsorized = summarize_significant_results(practice_results_iqr_winsorized, "Practice Comparison - Averaged, IQR Winsorized")

    print("\n" + "="*80)
    print("ANALYSIS COMPLETE")
    print("="*80)
    print(f"Check '{output_excel_file}' for detailed results.")
    print("The sheets ending with '_dataset_...' contain the final averaged data used for each analysis type.")

Original dataset: 540 rows, 45 unique participants
Total connectivity variables identified: 61
Averaging data across modes and states for each participant...
Grouping by: ['ID', 'Group']
Averaging 61 connectivity variables
Original data: 540 rows
Averaged data: 45 rows
Average rows per participant (before averaging): 12.00
Participant counts by group after averaging:
  YACs: 15 participants
  OACs: 15 participants
  TCOAs: 15 participants
RUNNING MANN-WHITNEY U TESTS WITH AVERAGED DATA - THREE APPROACHES

1. ORIGINAL ANALYSIS (With averaging across modes/states)

Performing Mann-Whitney U comparisons for Age Comparison - Averaged Data...
Groups in data: ['YACs' 'OACs']
Sample sizes: {'YACs': 15, 'OACs': 15}
Completed 61 comparisons for Age Comparison - Averaged Data

Performing Mann-Whitney U comparisons for Practice Comparison - Averaged Data...
Groups in data: ['OACs' 'TCOAs']
Sample sizes: {'OACs': 15, 'TCOAs': 15}
Completed 61 comparisons for Practice Comparison - Averaged Data

2.